# Perovskite Solar Cell Data — Exploration & Re-schema

โปรเจกต์นี้เน้น **กระบวนการ/ขั้นตอนการทดลอง (process)** ในการทำ perovskite solar cell
(ต่างจากงานที่เน้นค้นหาวัสดุ/ธาตุใหม่). Raw data อยู่ในโฟลเดอร์ `Raw Data/` โดย
**1 ไฟล์ `.xlsx` = 1 device (sample)** ตั้งชื่อว่า `<doi>_Sample<N>.xlsx`.

แต่ละ workbook มี 6 sheets: `Layer`, `Vender`, `Materials`, `Process`, `Properties`, `Stability`.

Notebook นี้แบ่งเป็น 3 ส่วน:
1. **Explore** — สำรวจไฟล์, ตรวจความสม่ำเสมอของ schema, หาปัญหา data quality
2. **Re-schema** — ออกแบบ + รัน ETL เปลี่ยนเป็นตาราง normalized ที่เชื่อมด้วย `device_id`

## Setup

In [1]:
import sys, os, glob, re
import pandas as pd

# ทำให้ import scripts/reschema.py ได้ ไม่ว่าจะรัน notebook จากที่ไหน
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd())=="notebooks" else "."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

RAW_DIR = "Raw Data"
files = sorted(glob.glob(os.path.join(RAW_DIR, "*.xlsx")))
print("repo root :", ROOT)
print("workbooks :", len(files))

repo root : d:\University\Year 4 Semester 1 (Intern)\Perovskites-Research
workbooks : 44


## Inventory — ไฟล์ / publication / sample

ชื่อไฟล์เข้ารหัส DOI + หมายเลข sample. แยกออกมาเป็นคีย์ที่ใช้งานได้.

In [2]:
from scripts.reschema import parse_device_id

inv = pd.DataFrame(
    [dict(zip(["device_id","doi_raw","sample_no"], parse_device_id(f))) | {"file": os.path.basename(f)}
     for f in files]
)
print(f"{len(inv)} devices จาก {inv['doi_raw'].nunique()} publications\n")
display(inv.head(10))
display(inv.groupby("doi_raw").size().rename("n_samples").sort_values(ascending=False))

44 devices จาก 15 publications



,device_id,doi_raw,sample_no,file
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,10.1002admi.202500231_Sample1.xlsx
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,10.1002admi.202500231_Sample2.xlsx
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,10.1002admi.202500231_Sample3.xlsx
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,10.1002admi.202500231_Sample4.xlsx
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,10.1016j.jechem.2021.11.029_Sample1.xlsx
5,10.1016j.jechem.2021.11.029_Sample2,10.1016j.jechem.2021.11.029,2,10.1016j.jechem.2021.11.029_Sample2.xlsx
6,10.1016j.nanoen.2021.106285_Sample1,10.1016j.nanoen.2021.106285,1,10.1016j.nanoen.2021.106285_Sample1.xlsx
7,10.1016j.nanoen.2021.106285_Sample2,10.1016j.nanoen.2021.106285,2,10.1016j.nanoen.2021.106285_Sample2.xlsx
8,10.1016j.nanoen.2021.106285_Sample3,10.1016j.nanoen.2021.106285,3,10.1016j.nanoen.2021.106285_Sample3.xlsx
9,10.1021_acsami.5c16972_Sample1,10.1021_acsami.5c16972,1,10.1021_acsami.5c16972_Sample1.xlsx


doi_raw
10_1002adma_202507730             5
10_1016_j_heliyon_2024_e25352     5
10.1002admi.202500231             4
10_1039_d3lf00258f                4
10_1021_acsami_5c15868            3
10.1016j.nanoen.2021.106285       3
10.1021_acsami.5c16972            3
10_31635_ccschem_025_202505977    3
10_1002_solr_202200422            2
10_1002_smll_202504237            2
10.1016j.jechem.2021.11.029       2
10_1002_solr_202400534            2
10_1016_j_solener_2025_113286     2
10_1021_acsami_4c14300            2
10_1126_science_ade9637           2
Name: n_samples, dtype: int64

## Schema audit — sheet และ header สม่ำเสมอแค่ไหน?

ตรวจว่าทุกไฟล์มี sheet/หัวคอลัมน์ตรงกันหรือไม่ (จุดที่ต้อง normalize).

In [3]:
from collections import Counter

sheet_combos, headers = Counter(), {}
for f in files:
    xl = pd.ExcelFile(f)
    sheet_combos[tuple(xl.sheet_names)] += 1
    for sh in xl.sheet_names:
        h = tuple(str(x).strip() for x in pd.read_excel(f, sh, nrows=0).columns)
        headers.setdefault(sh, Counter())[h] += 1

print("=== ชุดชื่อ sheet ที่พบ ===")
for combo, n in sheet_combos.most_common():
    print(f"  {n:2d}x  {combo}")

print("\n=== จำนวน header แบบที่ต่างกัน ต่อ sheet ===")
for sh, c in headers.items():
    print(f"  {sh:12s}: {len(c)} แบบ")

=== ชุดชื่อ sheet ที่พบ ===
  32x  ('Layer', 'Vender', 'Materials', 'Process', 'Properties', 'Stability')
   8x  ('Layer', 'Materials', 'Vender', 'Process', 'Properties', 'Stability')
   4x  ('Layer', 'Vender', 'Materials', 'Process', 'NiOx synthesis', 'Properties', 'Stability')

=== จำนวน header แบบที่ต่างกัน ต่อ sheet ===
  Layer       : 1 แบบ
  Vender      : 1 แบบ
  Materials   : 1 แบบ
  Process     : 1 แบบ
  NiOx synthesis: 1 แบบ
  Properties  : 2 แบบ
  Stability   : 8 แบบ


In [4]:
for h, n in headers["Properties"].most_common():
    print(f"{n:2d}x {h}")

31x ('Properties', 'Values', 'Errors', 'Unit', 'Remarks')
13x ('Properties', 'Values', 'Unit', 'Remarks')


In [ ]:
for h, n in headers["Stability"].most_common():
    print(f"{n:2d}x {h}")

27x ('Environment', 'Encapsulation', 'Temperature (°C)', 'Humidity (%)', 'Time (hr)', 'Retention (%)', 'T80 (hr)')
 4x ('Environment', 'Encapsulation', 'temperature (celsius)', 'relative humidity (%)', 'time (hr)', 'Normalize PCE (%)', 'Initial PCE (%)', 'Final PCE (%)')
 4x ('Environment', 'Encapsulation', 'temperature (C)', 'humidity (%)', 'time (hr)', 'Retention (%)')
 4x ('Environment', 'Encapsulation', 'Temperature (°C)', 'humidity (%)', 'time (hr)', 'Retention (%)')
 2x ('Environment', 'Encapsulation', 'temperature (celsius)', 'relative humidity (%)', 'time (hr)', 'Normalize PCE (%)')
 1x ('Environment', 'Encapsulation', 'temperature (C)', 'humidity (%)', 'time (hr)', 'Normalize PCE (%)')
 1x ('Environment', 'Encapsulation', 'temperature (celsius)', 'relative humidity (%)', 'time (hr)', 'Final normalize PCE (%)')
 1x ('Environment', 'Encapsulation', 'Temperature (°C)', 'Humidity (%)', 'Time (hr)', 'Retention (%)')


## ดูข้อมูลจริงของ 1 device (ครบ 6 sheets)

In [6]:
sample_file = files[0]
print("FILE:", os.path.basename(sample_file), "\n")
for sh in ["Layer","Vender","Materials","Process","Properties","Stability"]:
    print(f"───── {sh} ─────")
    display(pd.read_excel(sample_file, sh))

FILE: 10.1002admi.202500231_Sample1.xlsx 

───── Layer ─────


,LayerID,LayerType,Material,Thickness (nm)
0,1,Anode,ITO,NaN
1,2,HTL,NiOx NPs,NaN
2,3,HTL treatment,NaN,NaN
3,4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
4,5,PSK treatment,NaN,NaN
5,6,ETL,C60,50
6,7,ETL treatment,BCP,10
7,8,Cathode,Ag,100


───── Vender ─────


,Material,%Purity,Vender
0,ITO coated glass,9 - 15 Ω □-1,"Colorado Concept Coatings LLC, 96041"
1,DMF,99.9,Sigma-Aldrich
2,DMSO,99,Alfa Aesar
3,CB,99.8,Acros Organics
4,C60,99,MER Corporation
5,BCP,96,Sigma Aldrich
6,PbI2,99.99,TCI
7,CsI,99.999,Sigma Aldrich
8,MABr,>99.99,Greatcell Solar Materials
9,PbBr2,99.99,TCI


───── Materials ─────


,LayerID,Material,Type,Solvent,Amount,Unit
0,1,ITO,NaN,NaN,NaN,NaN
1,2,NiOx NPs,Solution,De-ionized water,22.0,mg/ml
2,3,NaN,NaN,NaN,NaN,NaN
3,4,PbI2,Solution,DMF/DMSO (4/1 V/V),324.0,mg
4,4,CsI,Solution,DMF/DMSO (4/1 V/V),11.0,mg
5,4,PbBr2,Solution,DMF/DMSO (4/1 V/V),47.0,mg
6,4,MABr,Solution,DMF/DMSO (4/1 V/V),14.0,NaN
7,4,FAI,Solution,DMF/DMSO (4/1 V/V),110.0,mg
8,5,NaN,NaN,NaN,NaN,NaN
9,6,C60,Solution,CB,NaN,NaN


... (2 outputs left out)

───── Properties ─────


,Properties,Values,Errors,Unit,Remarks
0,Voc,1.020,0.010,V,AM 1.5G
1,Jsc,20.970,1.130,mA cm−2,AM 1.5G
2,FF,0.760,0.005,NaN,AM 1.5G
3,PCE,16.310,1.000,%,AM 1.5G
4,Rs,0.680,NaN,Ohm/cm2,NaN
5,FWHM,0.144,0.079,eV,NaN
6,HTL optical band gap,3.850,NaN,eV,NiOx NPs
7,PSK optical band gap,1.609,NaN,eV,NaN
8,VBM,-5.720,NaN,eV,UPS
9,TRPL t1,6.000,1.000,ns,HTL/PSK half-stack


───── Stability ─────


,Environment,Encapsulation,temperature (celsius),relative humidity (%),time (hr),Normalize PCE (%),Initial PCE (%),Final PCE (%)
0,ISOS-L2,N,NaN,NaN,72,NaN,16.31,14.5


## Data-quality scan

ประเด็นที่เจอ และต้องจัดการตอน re-schema:
- ชื่อ `Properties` เป็น free-text หลากหลายมาก (200+ ชื่อ เช่น `Champion PCE`, `Avg PCE`, `PCE (RS)`)
  → เก็บแบบ **long / key–value (EAV)** ไม่ใช่ wide columns
- header ของ `Stability` สะกด/ใช้ตัวพิมพ์ต่างกัน → map เป็นชื่อมาตรฐาน
- มีเซลล์ว่าง, ช่องว่าง/ขึ้นบรรทัดปนในข้อความ, แถวว่าง → ต้อง clean

In [7]:
prop_names = Counter()
for f in files:
    s = pd.read_excel(f, "Properties")["Properties"].dropna().astype(str).str.strip()
    prop_names.update(s)
print(f"property names ที่ไม่ซ้ำกัน: {len(prop_names)} ชื่อ  (top 15)")
for name, n in prop_names.most_common(15):
    print(f"  {n:3d}  {name}")

property names ที่ไม่ซ้ำกัน: 109 ชื่อ  (top 15)
   26  VTFL
   24  Champion PCE
   22  Rs
   20  Champion Voc
   20  Champion Jsc
   20  Champion FF
   15  VBM
   15  Rrec
   13  Average PCE
   12  Average Voc
   12  Average Jsc
   12  Average FF
   11  HI
   10  Grain size
   10  Jsc EQE


## Re-schema — โมเดล normalized

ทุกตารางเชื่อมกันด้วยคีย์เดียว **`device_id`** (= ชื่อไฟล์ เช่น `10_1126_science_ade9637_Sample2`).

In [8]:
from importlib import reload
import scripts.reschema as rs
reload(rs)

tables = rs.build_tables()
for name, df in tables.items():
    print(f"{name:12s} {df.shape}")

devices      (44, 4)
layers       (352, 5)
vendors      (644, 4)
materials    (543, 7)
process      (702, 9)
properties   (563, 6)
stability    (62, 10)


In [9]:
for name in ["devices","layers","materials","process","properties","stability"]:
    print(f"\n───── {name} ─────")
    display(tables[name].head(6))


───── devices ─────


,device_id,doi_raw,sample_no,source_file
0,10.1002admi.202500231_Sample1,10.1002admi.202500231,1,10.1002admi.202500231_Sample1.xlsx
1,10.1002admi.202500231_Sample2,10.1002admi.202500231,2,10.1002admi.202500231_Sample2.xlsx
2,10.1002admi.202500231_Sample3,10.1002admi.202500231,3,10.1002admi.202500231_Sample3.xlsx
3,10.1002admi.202500231_Sample4,10.1002admi.202500231,4,10.1002admi.202500231_Sample4.xlsx
4,10.1016j.jechem.2021.11.029_Sample1,10.1016j.jechem.2021.11.029,1,10.1016j.jechem.2021.11.029_Sample1.xlsx
5,10.1016j.jechem.2021.11.029_Sample2,10.1016j.jechem.2021.11.029,2,10.1016j.jechem.2021.11.029_Sample2.xlsx



───── layers ─────


,device_id,layer_id,layer_type,material,thickness_nm
0,10.1002admi.202500231_Sample1,1,Anode,ITO,NaN
1,10.1002admi.202500231_Sample1,2,HTL,NiOx NPs,NaN
2,10.1002admi.202500231_Sample1,3,HTL treatment,NaN,NaN
3,10.1002admi.202500231_Sample1,4,PSK,[ (Cs.05FA.79MA.16)Pb(I.87Br.13)3 ],400-450
4,10.1002admi.202500231_Sample1,5,PSK treatment,NaN,NaN
5,10.1002admi.202500231_Sample1,6,ETL,C60,50



───── materials ─────


,device_id,layer_id,material,type,solvent,amount,unit
0,10.1002admi.202500231_Sample1,1.0,ITO,NaN,NaN,NaN,NaN
1,10.1002admi.202500231_Sample1,2.0,NiOx NPs,Solution,De-ionized water,22.0,mg/ml
2,10.1002admi.202500231_Sample1,3.0,NaN,NaN,NaN,NaN,NaN
3,10.1002admi.202500231_Sample1,4.0,PbI2,Solution,DMF/DMSO (4/1 V/V),324.0,mg
4,10.1002admi.202500231_Sample1,4.0,CsI,Solution,DMF/DMSO (4/1 V/V),11.0,mg
5,10.1002admi.202500231_Sample1,4.0,PbBr2,Solution,DMF/DMSO (4/1 V/V),47.0,mg


... (2 outputs left out)


───── properties ─────


,device_id,property,value,error,unit,remarks
0,10.1002admi.202500231_Sample1,Voc,1.020,0.010,V,AM 1.5G
1,10.1002admi.202500231_Sample1,Jsc,20.970,1.130,mA cm−2,AM 1.5G
2,10.1002admi.202500231_Sample1,FF,0.760,0.005,NaN,AM 1.5G
3,10.1002admi.202500231_Sample1,PCE,16.310,1.000,%,AM 1.5G
4,10.1002admi.202500231_Sample1,Rs,0.680,NaN,Ohm/cm2,NaN
5,10.1002admi.202500231_Sample1,FWHM,0.144,0.079,eV,NaN



───── stability ─────


,device_id,environment,encapsulation,temperature_c,humidity_pct,time_hr,retention_pct,t80_hr,initial_pce,final_pce
0,10.1002admi.202500231_Sample1,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.31,14.5
1,10.1002admi.202500231_Sample2,ISOS-L2,N,NaN,NaN,72,NaN,NaN,17.11,10.5
2,10.1002admi.202500231_Sample3,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.97,11.8
3,10.1002admi.202500231_Sample4,ISOS-L2,N,NaN,NaN,72,NaN,NaN,16.25,13.4
4,10.1016j.jechem.2021.11.029_Sample1,Nitrogen (N2),N,NaN,NaN,2880,86,NaN,NaN,NaN
5,10.1016j.jechem.2021.11.029_Sample1,Operational (MPP),N,NaN,NaN,700,66,NaN,NaN,NaN
